# Lesson 10 - उत्पादनातील AI एजंट्स

या धड्यात तुम्ही Microsoft Agent Framework (Python) वापरून AI एजंटसाठी **उत्पादन नमुने** शिकाल. आम्ही यात समाविष्ट करतो:

- **निरीक्षणक्षमता** — एजंट संवादांवर वेळ मोजणी आणि लॉगिंग जोडणे
- **मूल्यांकन** — प्रतिसादाच्या गुणवत्तेचे स्कोअर देण्यासाठी एक मूल्यांकन एजंट वापरणे
- **खर्च व्यवस्थापन** — टोकन ऑप्टिमायझेशन आणि मॉडेल निवडीसाठी धोरणे

घडामोडीचा दाखला एक **प्रवास एजंट** आहे जो वापरकर्त्यांना ट्रिप्स नियोजित करण्यात मदत करतो, ज्यावर निरीक्षण आणि मूल्यांकन लागू केलेले आहे.


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## उत्पादन विचार

AI एजंट्सना प्रोटोटाइप्समधून उत्पादनात नेण्यासाठी तीन मुख्य स्तंभांवर काळजीपूर्वक लक्ष देणे आवश्यक आहे:

1. **प्रेक्षणीयता** — एजंट काय करत आहे, त्याला किती वेळ लागतो, आणि कोणती साधने वापरतो यावर आपल्याला दृश्यता हवी असते. ट्रेसिंग आणि लॉगिंगशिवाय उत्पादनातील समस्या डिबग करणे जवळजवळ अशक्य आहे.

2. **मूल्यमापन** — स्वयंचलित गुणवत्ता तपासणी एजंटच्या प्रतिसादांना अचूक, पूर्ण आणि उपयुक्त ठेवण्याची खात्री करते. एक मूल्यमापन एजंट निश्चित केलेल्या निकषांवर प्रतिसादांचे गुणांकन करू शकतो.

3. **खर्च व्यवस्थापन** — टोकन वापर खर्चावर थेट परिणाम करतो. प्रॉम्प्ट ऑप्टिमायझेशन, मॉडेल निवड आणि कॅशिंगसारख्या धोरणांमुळे गुणवत्ता बिनधास्त ठेवत खर्चांचे नियंत्रण शक्य होते.


## निरीक्षणीय एजंट तयार करणे

आम्ही प्रवासाचे साधने परिभाषित करतो आणि एजंट कॉलसह टाइमिंग वेढलेले असते जेणेकरून आम्ही विलंब पाहू शकू. उत्पादनात तुम्ही OpenTelemetry किंवा तत्सम ट्रेसिंग बॅकएंडशी एकत्र करू शकता.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## मूल्यांकन नमुने

एक सामान्य उत्पादन नमुना म्हणजे दुसऱ्या एजंटचा **मूल्यमापक** म्हणून उपयोग करणे. मूल्यमापक प्राथमिक एजंटच्या उत्तराला पूर्णता, अचूकता, आणि उपयुक्तता यांसारख्या पूर्वनिर्धारित निकषांनुसार गुणांकन करतो.

हे सक्षम करते:
- वापरकर्त्यांपर्यंत उत्तर पोहोचण्यापूर्वी स्वयंचलित गुणवत्ता नियंत्रण
- जेव्हा प्रॉम्प्ट्स किंवा मॉडेल्स बदलतात तेव्हा परत न येणाऱ्या त्रुटीचे शोध घेणे
- एजंटच्या कार्यक्षमतेच्या सततच्या निरीक्षणासाठी


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## खर्च व्यवस्थापन धोरणे

निर्मिती AI एजंट्ससाठी खर्चावर नियंत्रण ठेवणे अत्यंत आवश्यक आहे. येथे मुख्य धोरणे आहेत:

| धोरण | वर्णन |
|---|---|
| **प्रॉम्प्ट ऑप्टिमायझेशन** | प्रणाली सूचनांना संक्षिप्त ठेवा. इनपुट टोकन कमी करण्यासाठी अतिरिक्त संदर्भ काढा. |
| **मॉडेल निवड** | वर्गीकरण किंवा माहिती काढण्यासाठी छोटे, स्वस्त मॉडेल्स (उदा. GPT-4o-mini) वापरा आणि जटिल विचारांसाठी मोठे मॉडेल्स राखून ठेवा. |
| **कॅशिंग** | टूल परिणाम आणि वारंवार विचारले जाणारे प्रश्न कॅश करा जेणेकरून अनावश्यक API कॉल टाळता येतील. |
| **टोकन बजेट** | अनपेक्षितपणे लांब उत्तर टाळण्यासाठी `max_tokens` मर्यादा सेट करा. |
| **बॅचिंग** | शक्य तिथे अनेक वापरकर्त्यांच्या प्रश्नांना एका API कॉलमध्ये गटबद्ध करा. |

प्रत्यक्षात, एक स्तरित दृष्टिकोन चांगले कार्य करतो: सरळ साधे विनंती जलद, स्वस्त मॉडेलकडे वळवा आणि फक्त जटिल प्रश्न फक्त क्षमतेने अधिक असलेल्या मॉडेलकडे पाठवा.


## सारांश

या धड्यात तुम्ही शिकले कसे:

1. एजंट संवादांमध्ये टाइमिंग आणि लॉगिंगसह **पर्यवेक्षण जोडायचे**, ज्यामुळे ट्रेसिंग आणि मॉनिटरिंगसाठी पाया तयार होतो.
2. एक मूल्यांकन एजंट वापरून एजंटच्या प्रतिसादांचे **स्वतःच मूल्यमापन करायचे**, जे पूर्णता, अचूकता आणि उपयुक्ततेसाठी गुणांकन करतो.
3. प्रॉम्प्ट ऑप्टिमायझेशन, मॉडेल निवड, कॅशिंग आणि टोकन बजेट यांद्वारे **खर्च व्यवस्थापन करायचे**.

हे प्रॉडक्शन पॅटर्न तुमचे AI एजंट विश्वसनीय, मोजता येण्यास सोपे आणि प्रमाणात किफायतशीर होण्यास मदत करतात.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
